In [2]:
# pip install kagglehub
# !pip install duckdb pandas pyarrow

In [1]:
import kagglehub, os

path = kagglehub.dataset_download("nicolaswattiez/skillbuilder-data-2009-2010")
print("Carpeta:", path)

for f in os.listdir(path):
    p = os.path.join(path, f)
    print(f, round(os.path.getsize(p) / 1e9, 2), "GB")

Carpeta: C:\Users\TB Val 03\.cache\kagglehub\datasets\nicolaswattiez\skillbuilder-data-2009-2010\versions\4
2012-2013-data-with-predictions-4-final.csv 3.01 GB


In [3]:
import os
import duckdb

base = r"C:\Users\TB Val 03\Desktop\PAULA PROYECTOS\MLproject\proyecto"

ruta_csv = r"C:\Users\TB Val 03\.cache\kagglehub\datasets\nicolaswattiez\skillbuilder-data-2009-2010\versions\4\2012-2013-data-with-predictions-4-final.csv"
ruta_parquet = os.path.join(base, "data", "assistments_2012.parquet")
ruta_tmp = os.path.join(base, "tmp").replace("\\", "/")

con = duckdb.connect()
con.execute("SET memory_limit='10GB'")
con.execute(f"SET temp_directory='{ruta_tmp}'")

con.execute(f"""
COPY (
    SELECT DISTINCT ON (problem_log_id)
        problem_log_id, user_id, assignment_id, sequence_id, student_class_id,
        school_id, teacher_id, start_time, end_time, ms_first_response,
        overlap_time, correct, hint_count, attempt_count, bottom_hint,
        first_action, original, problem_type, type, tutor_mode, position,
        "Average_confidence(FRUSTRATED)"    AS conf_frustrated,
        "Average_confidence(CONFUSED)"      AS conf_confused,
        "Average_confidence(CONCENTRATING)" AS conf_concentrating,
        "Average_confidence(BORED)"         AS conf_bored
    FROM read_csv('{ruta_csv}',
                  header=true, quote='"', escape='"',
                  parallel=false,
                  strict_mode=false,
                  ignore_errors=true, null_padding=true, sample_size=-1)
) TO '{ruta_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD);
""")
print("Parquet creado en:", ruta_parquet)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Parquet creado en: C:\Users\TB Val 03\Desktop\PAULA PROYECTOS\MLproject\proyecto\data\assistments_2012.parquet


In [4]:
import pandas as pd

df = pd.read_parquet(ruta_parquet)   # en vez de pd.read_csv(...)
df.head()

,problem_log_id,user_id,assignment_id,sequence_id,student_class_id,school_id,teacher_id,start_time,end_time,ms_first_response,...,first_action,original,problem_type,type,tutor_mode,position,conf_frustrated,conf_confused,conf_concentrating,conf_bored
0,140987162,93581,679210,38845,22755,5450.0,68444,2013-01-17 15:15:37,2013-01-17 15:15:59.157,22154,...,0,1,algebra,LinearSection,tutor,120,0.361323,0.000000,0.766925,0.442968
1,143881629,100568,784443,10597,23158,5406.0,59507,2013-04-26 10:06:52,2013-04-26 10:08:50.218,74908,...,0,1,algebra,MasterySection,tutor,0,0.445625,0.166667,0.766925,0.298956
2,137572567,105799,564188,2646,23158,5406.0,59507,2012-09-24 10:15:22,2012-09-24 10:15:53.758,31743,...,0,0,choose_1,RandomIterateSection,tutor,2,0.361323,0.000000,0.766925,0.000000
3,137278023,121532,560320,15043,23352,76.0,73040,2012-09-14 09:54:29,2012-09-14 09:54:31.893,2890,...,0,1,choose_1,RandomChildOrderSection,tutor,3,0.361323,0.000000,0.211268,0.000000
4,140552704,122134,653085,106531,24229,397.0,61291,2013-01-03 11:49:14,2013-01-03 11:51:31.148,137132,...,0,1,algebra,RandomChildOrderSection,tutor,8,0.361323,0.000000,0.766925,0.442968


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6123270 entries, 0 to 6123269
Data columns (total 25 columns):
 #   Column              Dtype         
---  ------              -----         
 0   problem_log_id      int64         
 1   user_id             int64         
 2   assignment_id       int64         
 3   sequence_id         int64         
 4   student_class_id    int64         
 5   school_id           float64       
 6   teacher_id          int64         
 7   start_time          datetime64[us]
 8   end_time            datetime64[us]
 9   ms_first_response   int64         
 10  overlap_time        int64         
 11  correct             float64       
 12  hint_count          int64         
 13  attempt_count       int64         
 14  bottom_hint         float64       
 15  first_action        int64         
 16  original            int64         
 17  problem_type        str           
 18  type                str           
 19  tutor_mode          str           
 20  position     

In [6]:
print(df.problem_log_id.is_unique)                       # debe ser True tras el DISTINCT
print(df.user_id.nunique(), "estudiantes |", df.student_class_id.nunique(), "clases |", df.school_id.nunique(), "escuelas")
print(df.start_time.min(), "->", df.end_time.max())
print((df.end_time < df.start_time).sum(), "filas con end_time < start_time")

True
46674 estudiantes | 2661 clases | 661 escuelas
2012-09-01 00:00:06 -> 2013-10-09 20:00:02.411000
727 filas con end_time < start_time


In [7]:
print(df.isna().sum()[df.isna().sum() > 0])              # nulos por columna
print(df["type"].value_counts(dropna=False))             # Mastery / Linear / Random
print(df["tutor_mode"].value_counts(dropna=False))
print(df["original"].value_counts())
print(df["first_action"].value_counts())
print(df["correct"].describe())
print(df[["conf_frustrated","conf_confused","conf_concentrating","conf_bored"]].describe())

school_id        157
bottom_hint    60348
dtype: int64
type
LinearSection              3349671
MasterySection             1868746
RandomChildOrderSection     875949
RandomIterateSection         21452
PlacementsSection             3947
ChooseConditionSection        3489
NumericLimitSection             16
Name: count, dtype: int64
tutor_mode
tutor    6120998
test        2272
Name: count, dtype: int64
original
1    5819737
0     303533
Name: count, dtype: int64
first_action
0    5782418
1     305009
2      35843
Name: count, dtype: int64
count    6.123270e+06
mean     6.768206e-01
std      4.674909e-01
min      0.000000e+00
25%      0.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: correct, dtype: float64
       conf_frustrated  conf_confused  conf_concentrating    conf_bored
count     6.123270e+06   6.123270e+06        6.123270e+06  6.123270e+06
mean      3.894586e-01   4.479487e-02        6.823843e-01  2.567723e-01
std       1.027662e-01   1.924793e-01

In [8]:
# 1. Volumen por mes
print(df.groupby(df.start_time.dt.to_period("M")).size())

# 2. Distribución por hora del día (para ver la zona horaria)
print(df.groupby(df.start_time.dt.hour).size())

start_time
2012-09    656874
2012-10    739653
2012-11    577141
2012-12    425493
2013-01    638082
2013-02    450011
2013-03    514865
2013-04    783423
2013-05    711939
2013-06    230084
2013-07    149522
2013-08    246183
Freq: M, dtype: int64
start_time
0      47227
1      27255
2      17467
3       9164
4       6147
5       8737
6      33802
7      94459
8     441476
9     499467
10    590426
11    490775
12    474472
13    533169
14    421038
15    398060
16    361119
17    310798
18    293266
19    304977
20    296843
21    235173
22    144895
23     83058
dtype: int64


In [9]:
# Pendiente de antes
print(df.groupby(df.start_time.dt.dayofweek).size())      # 0 = lunes
for c in ["conf_frustrated", "conf_confused", "conf_concentrating", "conf_bored"]:
    print(c, "->", df[c].nunique(), "valores distintos")
    print(df[c].value_counts(normalize=True).head(4).round(3), "\n")
print(df["correct"].round(2).value_counts().head(6))
print(df.groupby("first_action")[["hint_count", "attempt_count"]].mean())

# Duración de cada problema (segundos), sin las 727 filas incoherentes
d = df[df.end_time >= df.start_time].copy()
d["dur_s"] = (d.end_time - d.start_time).dt.total_seconds()
print(d["dur_s"].quantile([.5, .9, .99, .999]).round(1))

# Huecos entre problemas consecutivos de un mismo estudiante (minutos)
d = d.sort_values(["user_id", "start_time"])
d["gap_min"] = (d.start_time - d.groupby("user_id").end_time.shift()).dt.total_seconds() / 60
print(d["gap_min"].quantile([.5, .75, .9, .95, .99]).round(1))

start_time
0    1107634
1    1151305
2    1185391
3    1238729
4     872036
5     159846
6     408329
dtype: int64
conf_frustrated -> 831 valores distintos
conf_frustrated
0.361323    0.920
0.775000    0.029
0.805195    0.018
0.821192    0.006
Name: proportion, dtype: float64 

conf_confused -> 2476 valores distintos
conf_confused
0.000000    0.938
1.000000    0.013
0.978947    0.007
0.939394    0.004
Name: proportion, dtype: float64 

conf_concentrating -> 842 valores distintos
conf_concentrating
0.766925    0.798
0.336529    0.150
0.211268    0.008
0.551727    0.008
Name: proportion, dtype: float64 

conf_bored -> 11213 valores distintos
conf_bored
0.000000    0.443
0.442968    0.173
0.762689    0.053
0.226950    0.049
Name: proportion, dtype: float64 

correct
1.00    4141564
0.00    1976383
0.50       2395
0.75       1629
0.25       1211
0.98         24
Name: count, dtype: int64
              hint_count  attempt_count
first_action                           
0               0.222412